# Guest Waves

This notebook details the construction of 'Guest waves' for use in the targeted period tests.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt
import h5py
from pydmd import DMD, BOPDMD
from pydmd.preprocessing import hankel_preprocessing
from matplotlib.colors import BoundaryNorm


# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *
from src.msc_thesis.synFigures import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep
['/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc

# Defining Candidates:

The candidate mode numbers (i.e. modes for which we would like to run targeted tests) are:

- 6 (and 1 accidentally) (long period)
- 32 (medium period)
- 48 (short period)

For each of these modes:

- find the other phasor in the data set that is most similar and most disperate



In [3]:
nmax = 15

base_set = ['62', '6', '13', '48', '45', '1', '32']

candidate_mode_nums =  ['62','45']
mode_numbers = np.arange(1, 63, 1)
mode_numbers = [str(mode_num) for mode_num in mode_numbers]

# withdrawing and storing all complex phasors
gnm_phasor_dict = {}
for mode_key in mode_numbers:
    mode_info_i = Component_Load_SV(mode_key, nmax=15)
    gnm_phasor_dict[mode_key] = mode_info_i["gnm"]

In [4]:


sv_phasor_dict = {}

for mode_key in mode_numbers:

    mode_sv_phasor = A_15_r @ Truncate_Gauss_Coeffs(gnm_phasor_dict[mode_key], tmax=nmax).T
    sv_phasor_dict[mode_key] = mode_sv_phasor

cand_correlations = {}

for cand_num in candidate_mode_nums:

    phasor_cand = sv_phasor_dict[cand_num]

    corr_n_list_sub = []

    cand_correlations[cand_num] = {}

    for compare_num in mode_numbers:

        if cand_num != compare_num:
            phasor_compare = sv_phasor_dict[compare_num]

            corr_compare = Complex_Phasor_Compare(
                phasor_cand, phasor_compare
            )
            cand_correlations[cand_num][compare_num] = corr_compare


In [5]:
print(cand_correlations)

{'62': {'1': np.float64(4.579915681584231e-10), '2': np.float64(0.0323031649780842), '3': np.float64(0.03683811021297599), '4': np.float64(0.04511295248496563), '5': np.float64(7.792161001489376e-10), '6': np.float64(1.1831838361577084e-09), '7': np.float64(0.1356396645032078), '8': np.float64(5.915650974630398e-10), '9': np.float64(0.023445828826621447), '10': np.float64(0.03275186876530532), '11': np.float64(7.272458047107832e-10), '12': np.float64(0.010987020790715256), '13': np.float64(0.01102745716179705), '14': np.float64(5.466178153016408e-10), '15': np.float64(0.02285667229794405), '16': np.float64(6.695193452224263e-10), '17': np.float64(0.07748383668619757), '18': np.float64(6.843136934584689e-10), '19': np.float64(3.828888520453394e-10), '20': np.float64(0.17654373289820294), '21': np.float64(0.266251968302075), '22': np.float64(2.866151112320746e-09), '23': np.float64(1.4412803594021395e-09), '24': np.float64(0.11908809206699116), '25': np.float64(2.7646999179838947e-09), '

In [6]:
guest_phasor_dict = {}

moderate_corr = 0.5

for cand_num in candidate_mode_nums:
    print(f"for candidate {cand_num}")
    guest_phasor_dict[cand_num] = {}


    lowest_corr = 1
    moderate_corr_dist = 0.9

    for compare_num in mode_numbers:
        if compare_num not in base_set:

            corr_compare = cand_correlations[cand_num][compare_num] 

            if corr_compare < lowest_corr:
                lowest_corr = corr_compare
                guest_phasor_dict[cand_num]["disparate"] = compare_num

            if np.abs(moderate_corr - corr_compare) < moderate_corr_dist:
                moderate_corr_dist = np.abs(moderate_corr - corr_compare)
                guest_phasor_dict[cand_num]["moderate"] = compare_num
                print(corr_compare)
            


    guest_phasor_dict[cand_num]["closest"] = cand_num

for candidate 62
0.0323031649780842
0.03683811021297599
0.04511295248496563
0.1356396645032078
0.17654373289820294
0.266251968302075
0.40836052844346843
0.42809110217790347
0.43541803299578935
for candidate 45
4.399906400446636e-09
8.308290040136298e-09
0.08673900665116521
0.1256040414224871
0.18193868619359443
0.22753998606489767
0.276072266440859
0.3370008964629143
0.3653902552580714


In [7]:
print(guest_phasor_dict)

{'62': {'disparate': '38', 'moderate': '59', 'closest': '62'}, '45': {'disparate': '4', 'moderate': '34', 'closest': '45'}}


# Constructing their properties:

The guest waves must now be ran through the resolution matrix procedure to obtain a scaled/ resolved gauss-spline representation. Here, we essentially create a secondary data base of 'guest waves' that can be read in as any other waves

In [8]:
# first adding the 'guest waves' to data base

# basic mode i loading code - assumes SV wanted
def Synthetic_raw_G_MF_Load(mode_number, directory=FELIX_DIR, nmax = 20):

    # select a mode number and corresponding file
    file = h5py.File(f'{directory}/mode_surface_including_gnm_{mode_number}.h5',"r")

    # gauss coefficient (magnetic scalar potential) phasors
    gnm_60 =  (np.asarray(file["gnmr"]) +\
         1j*np.asarray(file["gnmi"])) # [V_alfven, arbitrary]
    
    gnm = Truncate_Gauss_Coeffs(gnm_60, tmax=nmax)

    # loading in true period and decay rate
    omega = file["omega"][()] # angular frequency (rad/year)

    eigenvalue = 0 + 1j * omega # storing as eigenvalue - NO DECAY

    file.close() # close file

    mode_i_info = {
        "mode_number": mode_number, # mode number
        "gnm": gnm, # main field gnm phasor (Valfven, unscaled)
        "eigenvalue": eigenvalue, # no decay eigenvalue
    }

    # returns all basic components for the mode
    return mode_i_info


# defining the period perturbations at which guests should be stored
guest_period_perturbations = [-50, -45, -40, -35, -30, -25, -20, -15, -10, -5, -2, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

guest_dict = {}

# for each candidate
for cand_num in candidate_mode_nums:

    # load in candidate data
    cand_info = Synthetic_raw_G_MF_Load(cand_num)

    cand_eigenvalue = cand_info["eigenvalue"]
    cand_omega = cand_eigenvalue.imag
    cand_period = (2*np.pi)/(cand_omega)

    cand_guest_phasor_dict = guest_phasor_dict[cand_num]

    for guest_type in cand_guest_phasor_dict:
        guest_num = cand_guest_phasor_dict[guest_type]

        # get specified guest phasor
        guest_phasor = Synthetic_raw_G_MF_Load(guest_num)["gnm"]

        for period_perturb in guest_period_perturbations:
            guest_period = cand_period + (period_perturb / 100) * cand_period
            guest_omega = (2*np.pi)/(guest_period)

            label = f"mode_surface_including_gnm_g{cand_num}_{guest_type}_T{period_perturb}"

            guest_dict[label] = {
                "gnmr": guest_phasor.real,
                "gnmi": guest_phasor.imag,
                "omega": guest_omega
            }


In [9]:
print(guest_dict.keys())

dict_keys(['mode_surface_including_gnm_g62_disparate_T-50', 'mode_surface_including_gnm_g62_disparate_T-45', 'mode_surface_including_gnm_g62_disparate_T-40', 'mode_surface_including_gnm_g62_disparate_T-35', 'mode_surface_including_gnm_g62_disparate_T-30', 'mode_surface_including_gnm_g62_disparate_T-25', 'mode_surface_including_gnm_g62_disparate_T-20', 'mode_surface_including_gnm_g62_disparate_T-15', 'mode_surface_including_gnm_g62_disparate_T-10', 'mode_surface_including_gnm_g62_disparate_T-5', 'mode_surface_including_gnm_g62_disparate_T-2', 'mode_surface_including_gnm_g62_disparate_T2', 'mode_surface_including_gnm_g62_disparate_T5', 'mode_surface_including_gnm_g62_disparate_T10', 'mode_surface_including_gnm_g62_disparate_T15', 'mode_surface_including_gnm_g62_disparate_T20', 'mode_surface_including_gnm_g62_disparate_T25', 'mode_surface_including_gnm_g62_disparate_T30', 'mode_surface_including_gnm_g62_disparate_T35', 'mode_surface_including_gnm_g62_disparate_T40', 'mode_surface_includin

In [10]:
# Save each guest wave directly inside modes_surface.
guest_output_directory = Path(FELIX_DIR)
expected_guest_fields = {"gnmr", "gnmi", "omega"}
expected_guest_coefficient_count = n_Gauss_Coeffs(20)

guest_output_paths = {
    label: guest_output_directory / f"{label}.h5"
    for label in guest_dict
}
guest_temporary_paths = {
    label: output_path.with_name(
        f".{output_path.stem}.incomplete{output_path.suffix}"
    )
    for label, output_path in guest_output_paths.items()
}

existing_paths = [
    path
    for path in (
        *guest_output_paths.values(),
        *guest_temporary_paths.values(),
    )
    if path.exists()
]
'''if existing_paths:
    raise FileExistsError(
        "Refusing to overwrite existing guest-wave files: "
        + ", ".join(str(path) for path in existing_paths)
    )'''

for label, guest_data in guest_dict.items():
    if set(guest_data) != expected_guest_fields:
        raise ValueError(
            f"{label} contains fields {set(guest_data)}; expected "
            f"{expected_guest_fields}."
        )

    gnmr = np.asarray(guest_data["gnmr"], dtype=float)
    gnmi = np.asarray(guest_data["gnmi"], dtype=float)
    omega = np.asarray(guest_data["omega"], dtype=float)

    expected_shape = (expected_guest_coefficient_count,)
    if gnmr.shape != expected_shape or gnmi.shape != expected_shape:
        raise ValueError(
            f"{label} has coefficient shapes {gnmr.shape} and "
            f"{gnmi.shape}; expected {expected_shape}."
        )
    if omega.shape != ():
        raise ValueError(f"{label} omega must be scalar, not {omega.shape}.")
    if not (
        np.all(np.isfinite(gnmr))
        and np.all(np.isfinite(gnmi))
        and np.isfinite(omega)
        and omega > 0
    ):
        raise ValueError(f"{label} contains invalid guest-wave data.")

    temporary_path = guest_temporary_paths[label]
    output_path = guest_output_paths[label]

    with h5py.File(temporary_path, "x") as guest_file:
        guest_file.create_dataset("gnmr", data=gnmr)
        guest_file.create_dataset("gnmi", data=gnmi)
        guest_file.create_dataset("omega", data=omega)

    with h5py.File(temporary_path, "r") as guest_file:
        if set(guest_file.keys()) != expected_guest_fields:
            raise ValueError(f"Written datasets do not match for {label}.")
        for field in expected_guest_fields:
            if not np.array_equal(guest_file[field][()], guest_data[field]):
                raise ValueError(
                    f"Read-back validation failed for {label}/{field}."
                )

    temporary_path.rename(output_path)

print(
    f"Saved and validated {len(guest_output_paths)} guest-wave files "
    f"in {guest_output_directory}."
)

Saved and validated 132 guest-wave files in /home/elliott/projects/msc_thesis_project/data/modes_surface.


# Resolve guest waves

Construct the physical-unit, unscaled, no-decay MF B-spline representation for every guest wave. The MF phasors are evaluated over the complete resolution interval, projected with `P`, and passed through the same MF resolution matrix used for the original modes. Only the real `without_decay` spline is retained.

In [11]:
import shutil

GUEST_NMAX = 20
GUEST_RAW_PREFIX = "mode_surface_including_gnm_"
EXPECTED_TIME_SAMPLES = 148
EXPECTED_SPLINE_SHAPE = (64, n_Gauss_Coeffs(GUEST_NMAX))
EXPECTED_RESOLUTION_DIMENSION = int(np.prod(EXPECTED_SPLINE_SHAPE))

spline_path = Path(FELIX_DIR) / "R_splines_arbitrary.h5"
staged_spline_path = spline_path.with_name(
    "R_splines_arbitrary.with_guests.incomplete.h5"
)
backup_spline_path = spline_path.with_name(
    "R_splines_arbitrary.pre_guest_backup.h5"
)
resolution_path = (
    Path(CHAOS_RESOL_DIR) / "CHAOS_Resol_1997_2026_0806_MF.h5"
)

guest_mode_ids = [
    label.removeprefix(GUEST_RAW_PREFIX)
    for label in guest_dict
]
expected_guest_groups = {
    f"mode_{guest_mode_id}"
    for guest_mode_id in guest_mode_ids
}

'''if len(guest_mode_ids) != 60 or len(set(guest_mode_ids)) != 60:
    raise ValueError("Expected exactly 60 unique guest mode identifiers.")
if np.shape(times_res_relative) != (EXPECTED_TIME_SAMPLES,):
    raise ValueError("The resolution time convention has changed.")
if not np.isclose(times_res_relative[0], 0.0):
    raise ValueError("Resolution-relative time must start at zero.")
if P.shape != (EXPECTED_SPLINE_SHAPE[0], EXPECTED_TIME_SAMPLES):
    raise ValueError(f"Unexpected MF projection shape: {P.shape}.")
if not spline_path.is_file() or not resolution_path.is_file():
    raise FileNotFoundError("Required spline or resolution file is missing.")
if staged_spline_path.exists() or backup_spline_path.exists():
    raise FileExistsError(
        "Refusing to overwrite a staged spline file or retained backup."
    )'''

missing_guest_files = [
    Path(FELIX_DIR) / f"{GUEST_RAW_PREFIX}{guest_mode_id}.h5"
    for guest_mode_id in guest_mode_ids
    if not (
        Path(FELIX_DIR) / f"{GUEST_RAW_PREFIX}{guest_mode_id}.h5"
    ).is_file()
]
if missing_guest_files:
    raise FileNotFoundError(
        "Missing guest files: "
        + ", ".join(str(path) for path in missing_guest_files)
    )

with h5py.File(spline_path, "r") as spline_file:
    original_groups = set(spline_file.keys())
    collisions = original_groups & expected_guest_groups
'''    if collisions:
        raise ValueError(f"Guest groups already exist: {sorted(collisions)}")'''

with h5py.File(resolution_path, "r") as resolution_file:
    resolution_shape = resolution_file["R"].shape
if resolution_shape != (
    EXPECTED_RESOLUTION_DIMENSION,
    EXPECTED_RESOLUTION_DIMENSION,
):
    raise ValueError(f"Unexpected resolution shape: {resolution_shape}.")

print(
    f"Validated {len(guest_mode_ids)} guest inputs; expected resolved "
    f"shape is {EXPECTED_SPLINE_SHAPE}."
)

Validated 132 guest inputs; expected resolved shape is (64, 440).


In [12]:
# Construct all physical-unit, unscaled MF spline inputs.
guest_spline_inputs = {}

for guest_mode_id in guest_mode_ids:
    mode_data = Component_Load_MF(
        guest_mode_id,
        nmax=GUEST_NMAX,
    )
    mf_phasor = np.asarray(mode_data["gnm"])
    eigenvalue = complex(np.asarray(mode_data["eigenvalue"]).item())

    if mf_phasor.shape != (EXPECTED_SPLINE_SHAPE[1],):
        raise ValueError(
            f"{guest_mode_id} has MF phasor shape {mf_phasor.shape}."
        )
    if not np.all(np.isfinite(mf_phasor)):
        raise ValueError(f"{guest_mode_id} has a non-finite MF phasor.")
    if (
        not np.isfinite(eigenvalue)
        or eigenvalue.real != 0.0
        or eigenvalue.imag <= 0.0
    ):
        raise ValueError(f"{guest_mode_id} has invalid eigenvalue {eigenvalue}.")

    mf_time_series = G_Time_Series_Eval(
        mf_phasor,
        eigenvalue,
        times=np.asarray(times_res_relative),
    )
    if mf_time_series.shape != (
        EXPECTED_TIME_SAMPLES,
        EXPECTED_SPLINE_SHAPE[1],
    ):
        raise ValueError(
            f"{guest_mode_id} produced time-series shape "
            f"{mf_time_series.shape}."
        )

    mf_spline = P @ mf_time_series
    if mf_spline.shape != EXPECTED_SPLINE_SHAPE:
        raise ValueError(
            f"{guest_mode_id} produced spline shape {mf_spline.shape}."
        )
    if np.iscomplexobj(mf_spline) or not np.all(np.isfinite(mf_spline)):
        raise ValueError(f"{guest_mode_id} produced an invalid MF spline.")

    guest_spline_inputs[guest_mode_id] = np.asarray(mf_spline)

guest_spline_input_matrix = np.column_stack([
    guest_spline_inputs[guest_mode_id].ravel(order="F")
    for guest_mode_id in guest_mode_ids
])
if guest_spline_input_matrix.shape != (
    EXPECTED_RESOLUTION_DIMENSION,
    len(guest_mode_ids),
):
    raise ValueError("Unexpected batched spline-input shape.")

print(f"Constructed input matrix {guest_spline_input_matrix.shape}.")

Constructed input matrix (28160, 132).


In [13]:
# Apply the same MF resolution matrix to all guests in one blockwise pass.
resolution_row_block_size = 440
guest_resolved_matrix = np.empty_like(guest_spline_input_matrix)

with h5py.File(resolution_path, "r") as resolution_file:
    resolution_matrix = resolution_file["R"]

    for row_start in range(
        0,
        EXPECTED_RESOLUTION_DIMENSION,
        resolution_row_block_size,
    ):
        row_stop = min(
            row_start + resolution_row_block_size,
            EXPECTED_RESOLUTION_DIMENSION,
        )
        resolution_block = np.asarray(
            resolution_matrix[row_start:row_stop, :]
        )
        guest_resolved_matrix[row_start:row_stop, :] = (
            resolution_block @ guest_spline_input_matrix
        )
        print(
            f"Resolved rows {row_start}:{row_stop} / "
            f"{EXPECTED_RESOLUTION_DIMENSION}"
        )

guest_resolved_splines = {
    guest_mode_id: guest_resolved_matrix[:, guest_index].reshape(
        EXPECTED_SPLINE_SHAPE,
        order="F",
    )
    for guest_index, guest_mode_id in enumerate(guest_mode_ids)
}

for guest_mode_id, resolved_spline in guest_resolved_splines.items():
    if resolved_spline.shape != EXPECTED_SPLINE_SHAPE:
        raise ValueError(f"Unexpected resolved shape for {guest_mode_id}.")
    if (
        np.iscomplexobj(resolved_spline)
        or not np.all(np.isfinite(resolved_spline))
    ):
        raise ValueError(f"Invalid resolved spline for {guest_mode_id}.")

print(f"Resolved and validated {len(guest_resolved_splines)} guests.")

Resolved rows 0:440 / 28160
Resolved rows 440:880 / 28160
Resolved rows 880:1320 / 28160
Resolved rows 1320:1760 / 28160
Resolved rows 1760:2200 / 28160
Resolved rows 2200:2640 / 28160
Resolved rows 2640:3080 / 28160
Resolved rows 3080:3520 / 28160
Resolved rows 3520:3960 / 28160
Resolved rows 3960:4400 / 28160
Resolved rows 4400:4840 / 28160
Resolved rows 4840:5280 / 28160
Resolved rows 5280:5720 / 28160
Resolved rows 5720:6160 / 28160
Resolved rows 6160:6600 / 28160
Resolved rows 6600:7040 / 28160
Resolved rows 7040:7480 / 28160
Resolved rows 7480:7920 / 28160
Resolved rows 7920:8360 / 28160
Resolved rows 8360:8800 / 28160
Resolved rows 8800:9240 / 28160
Resolved rows 9240:9680 / 28160
Resolved rows 9680:10120 / 28160
Resolved rows 10120:10560 / 28160
Resolved rows 10560:11000 / 28160
Resolved rows 11000:11440 / 28160
Resolved rows 11440:11880 / 28160
Resolved rows 11880:12320 / 28160
Resolved rows 12320:12760 / 28160
Resolved rows 12760:13200 / 28160
Resolved rows 13200:13640 / 2816

In [14]:
# Explicitly define groups that must never be overwritten.
mode_numbers = np.arange(1, 63, 1)
mode_numbers = [str(mode_number) for mode_number in mode_numbers]
protected_groups = {
    f"mode_{mode_id}"
    for mode_id in mode_numbers
}

expected_guest_groups = {
    f"mode_{guest_mode_id}"
    for guest_mode_id in guest_mode_ids
}

# Prevent accidental overlap between guest IDs and Felix IDs.
overlap = protected_groups & expected_guest_groups
if overlap:
    raise ValueError(
        f"Guest groups overlap protected Felix groups: {sorted(overlap)}"
    )

# Append to a staged copy.
shutil.copy2(spline_path, staged_spline_path)

with h5py.File(staged_spline_path, "r+") as staged_file:
    for guest_mode_id in guest_mode_ids:
        group_name = f"mode_{guest_mode_id}"

        # Existing guest groups may be replaced.
        if group_name in staged_file:
            if group_name in protected_groups:
                raise ValueError(
                    f"Refusing to overwrite protected group {group_name}."
                )

            del staged_file[group_name]

        group = staged_file.create_group(group_name)

        group.create_dataset(
            "without_decay",
            data=guest_resolved_splines[guest_mode_id],
        )

    staged_file.flush()


# Confirm that protected Felix datasets are unchanged and guests are correct.
with (
    h5py.File(spline_path, "r") as original_file,
    h5py.File(staged_spline_path, "r") as staged_file,
):
    # Ensure every protected group is still present.
    missing_protected = protected_groups - set(staged_file.keys())

    if missing_protected:
        raise ValueError(
            f"Protected groups are missing: {sorted(missing_protected)}"
        )

    # Check only protected Felix groups for unchanged data.
    for group_name in protected_groups:
        if group_name not in original_file:
            raise ValueError(
                f"Protected group {group_name} was not in the original file."
            )

        if set(staged_file[group_name].keys()) != set(
            original_file[group_name].keys()
        ):
            raise ValueError(
                f"Datasets changed in protected group {group_name}."
            )

        for dataset_name in original_file[group_name]:
            if not np.array_equal(
                staged_file[group_name][dataset_name][()],
                original_file[group_name][dataset_name][()],
            ):
                raise ValueError(
                    f"Original data changed in "
                    f"{group_name}/{dataset_name}."
                )

    # Check newly written or overwritten guest groups.
    for guest_mode_id in guest_mode_ids:
        group_name = f"mode_{guest_mode_id}"

        if set(staged_file[group_name].keys()) != {"without_decay"}:
            raise ValueError(
                f"Unexpected guest datasets in {group_name}."
            )

        dataset = staged_file[group_name]["without_decay"]

        if dataset.shape != EXPECTED_SPLINE_SHAPE:
            raise ValueError(
                f"Unexpected stored shape in {group_name}: "
                f"{dataset.shape}"
            )

        if not np.array_equal(
            dataset[()],
            guest_resolved_splines[guest_mode_id],
        ):
            raise ValueError(
                f"Guest read-back failed for {group_name}."
            )


# Replace any previous backup if appropriate.
if backup_spline_path.exists():
    backup_spline_path.unlink()

spline_path.rename(backup_spline_path)
staged_spline_path.rename(spline_path)

with h5py.File(spline_path, "r") as final_file:
    final_guest_groups = (
        set(final_file.keys()) & expected_guest_groups
    )

    if final_guest_groups != expected_guest_groups:
        raise ValueError("Final guest-group validation failed.")

print(
    f"Promoted validated spline file with "
    f"{len(expected_guest_groups)} guest groups. "
    f"Original retained at {backup_spline_path}."
)

Promoted validated spline file with 132 guest groups. Original retained at /home/elliott/projects/msc_thesis_project/data/modes_surface/R_splines_arbitrary.pre_guest_backup.h5.
